In [43]:
# Recalculate candidate S1 coverage from the CURRENT candidate file

current_candidate_s1 = set(
    candidates["source1_entity_id"].dropna().unique()
)

true_match_s1 = set(
    ground_truth.loc[
        ground_truth["source2_entity_id"].ne("") |
        ground_truth["source3_entity_id"].ne(""),
        "source1_entity_id"
    ].unique()
)

covered_true_s1 = true_match_s1 & current_candidate_s1

print("Current candidate S1s:", len(current_candidate_s1))
print("TRUE-match S1s:", len(true_match_s1))
print("TRUE-match S1s covered:", len(covered_true_s1))
print(
    "TRUE-match S1 coverage:",
    round(100 * len(covered_true_s1) / len(true_match_s1), 2),
    "%"
)

KeyError: 'source2_entity_id'

In [44]:
print("Ground truth columns:")
print(ground_truth.columns.tolist())

print("\nCandidate columns:")
print(candidates.columns.tolist())

Ground truth columns:
['source1_entity_id', 'matched_entity_ids', 'has_match']

Candidate columns:
['source1_entity_id', 'candidate_entity_id', 'candidate_source']


In [45]:
# Correct coverage calculation using your actual column names

current_candidate_s1 = set(
    candidates["source1_entity_id"].dropna().unique()
)

true_match_s1 = set(
    ground_truth.loc[
        ground_truth["has_match"].astype(str).str.lower().isin(["true", "1", "yes"]),
        "source1_entity_id"
    ].unique()
)

covered_true_s1 = true_match_s1 & current_candidate_s1

print("Current candidate S1s:", len(current_candidate_s1))
print("TRUE-match S1s:", len(true_match_s1))
print("TRUE-match S1s covered:", len(covered_true_s1))
print(
    "TRUE-match S1 coverage:",
    round(100 * len(covered_true_s1) / len(true_match_s1), 2),
    "%"
)

Current candidate S1s: 461895
TRUE-match S1s: 2083574
TRUE-match S1s covered: 439451
TRUE-match S1 coverage: 21.09 %


In [42]:
test_s1 = test_row["entity_id"]

print("Checking candidate file for:", test_s1)

matched_rows = candidates[
    candidates["source1_entity_id"] == test_s1
]

print("Candidate rows found:", len(matched_rows))

display(matched_rows.head(20))

Checking candidate file for: S1-925783039
Candidate rows found: 174


,source1_entity_id,candidate_entity_id,candidate_source
0,S1-925783039,S2-252379152,source2
1,S1-925783039,S2-335180728,source2
2,S1-925783039,S3-534582038,source3
3,S1-925783039,S2-68778321,source2
4,S1-925783039,S2-141689655,source2
5,S1-925783039,S2-138574636,source2
6,S1-925783039,S2-597503035,source2
7,S1-925783039,S2-371520891,source2
8,S1-925783039,S3-794815197,source3
9,S1-925783039,S3-954751262,source3


In [46]:
print("Candidate S1 count:", candidates["source1_entity_id"].nunique())

print(
    "Ground-truth S1 count:",
    ground_truth["source1_entity_id"].nunique()
)

print(
    "Ground-truth matched S1 count:",
    ground_truth.loc[
        ground_truth["has_match"] == True,
        "source1_entity_id"
    ].nunique()
)

Candidate S1 count: 461895
Ground-truth S1 count: 2206821
Ground-truth matched S1 count: 2083574


In [47]:
# Read ONLY the S1 IDs from the current candidate file, in chunks.
# This avoids loading all 186M candidate rows into memory.

candidate_s1_new = set()

for chunk in pd.read_csv(
    candidate_path,
    sep="\t",
    dtype=str,
    header=None,
    names=["source1_entity_id", "candidate_entity_id", "candidate_source"],
    usecols=["source1_entity_id"],
    chunksize=500_000,
    keep_default_na=False
):
    candidate_s1_new.update(chunk["source1_entity_id"].unique())

print("Current candidate S1 count:", len(candidate_s1_new))

true_match_s1 = set(
    ground_truth.loc[
        ground_truth["has_match"] == True,
        "source1_entity_id"
    ].unique()
)

covered_true_s1 = true_match_s1 & candidate_s1_new

print("TRUE-match S1 count:", len(true_match_s1))
print("TRUE-match S1s covered:", len(covered_true_s1))
print(
    "TRUE-match S1 coverage:",
    round(100 * len(covered_true_s1) / len(true_match_s1), 2),
    "%"
)

Current candidate S1 count: 1161324
TRUE-match S1 count: 2083574
TRUE-match S1s covered: 1105360
TRUE-match S1 coverage: 53.05 %


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

TRAIN_DIR = PROJECT_ROOT / "data" / "raw" / "train"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

S1_PATH = TRAIN_DIR / "train_source1.tsv"
S2_PATH = TRAIN_DIR / "train_source2.tsv"
S3_PATH = TRAIN_DIR / "train_source3.tsv"
GROUND_TRUTH_PATH = TRAIN_DIR / "train_ground_truth.tsv"
CANDIDATE_PATH = PROCESSED_DIR / "candidate_pairs.tsv"

print("S1:", S1_PATH)
print("S2:", S2_PATH)
print("S3:", S3_PATH)
print("Ground truth:", GROUND_TRUTH_PATH)
print("Candidates:", CANDIDATE_PATH)

print("\nCandidate file exists:", CANDIDATE_PATH.exists())

In [41]:
# Check the blocking index correctly

test_row = sample_missing.iloc[0]

print("S1:", test_row["entity_id"])
print("Name:", test_row["business_name"])
print("Country:", test_row["country"])
print("Tokens:", test_row["name_tokens"])

country = normalize_text(test_row["country"])

for token in test_row["name_tokens"]:
    rows = conn_check.execute(
        """
        SELECT source, entity_id
        FROM token_index
        WHERE country = ? AND token = ?
        LIMIT 10
        """,
        (country, token)
    ).fetchall()

    print(f"\nTOKEN: {token}")
    print("Matches in index:", len(rows))
    for r in rows[:5]:
        print(r)

S1: S1-925783039
Name: Orelee's Barbershop
Country: US
Tokens: {'barbershop', 'orelee'}

TOKEN: barbershop
Matches in index: 0

TOKEN: orelee
Matches in index: 10
('source2', 'S2-361957120')
('source2', 'S2-162361984')
('source2', 'S2-141689655')
('source2', 'S2-137340653')
('source2', 'S2-100198226')


In [40]:
import sqlite3

DB_FILE = PROCESSED_DIR / "blocking_index.db"

conn_check = sqlite3.connect(DB_FILE)

test_row = sample_missing.iloc[0]

print("S1:", test_row["entity_id"])
print("Name:", test_row["business_name"])
print("Country:", test_row["country"])
print("Tokens:", test_row["name_tokens"])

country = normalize_text(test_row["country"])

for token in test_row["name_tokens"]:
    rows = conn_check.execute(
        """
        SELECT source, entity_id, name
        FROM token_index
        WHERE country = ? AND token = ?
        LIMIT 10
        """,
        (country, token)
    ).fetchall()

    print(f"\nTOKEN: {token}")
    print("Matches in index:", len(rows))
    for r in rows[:5]:
        print(r)

S1: S1-925783039
Name: Orelee's Barbershop
Country: US
Tokens: {'barbershop', 'orelee'}


OperationalError: no such column: name

In [38]:
# Check one missed S1 directly against the SQLite blocking index

test_row = sample_missing.iloc[0]

print("S1:", test_row["entity_id"])
print("Name:", test_row["business_name"])
print("Country:", test_row["country"])
print("Tokens:", test_row["name_tokens"])

country = normalize_text(test_row["country"])

for token in test_row["name_tokens"]:
    rows = conn_check.execute(
        """
        SELECT source, entity_id, name
        FROM token_index
        WHERE country = ? AND token = ?
        LIMIT 10
        """,
        (country, token)
    ).fetchall()

    print(f"\nTOKEN: {token}")
    print("Matches in index:", len(rows))
    for r in rows[:5]:
        print(r)

S1: S1-925783039
Name: Orelee's Barbershop
Country: US
Tokens: {'barbershop', 'orelee'}


ProgrammingError: Cannot operate on a closed database.

In [36]:
sample_missing = missing_df[
    missing_df["has_index_token"]
].head(20)

display(
    sample_missing[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country",
            "name_tokens"
        ]
    ]
)

,entity_id,business_name,business_address,country,name_tokens
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,"{barbershop, orelee}"
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,"{prabhav, business}"
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India,"{consulting, private, nyasa, nursing}"
18,S1-221443075,Iris Advisors,House No-777 Raghubir Bhawanmadan Pur Khadar S...,India,"{iris, advisors}"
23,S1-716574008,Rajani Enterprises of Kannur,"46/1106, Dhanalakshmi Road, Near Amani Auditor...",India,"{enterprises, rajani, kannur}"
24,S1-985038901,"Schaefer, Michael and Silver Associates","32876 Circle Drive, Millsboro, DE",US,"{silver, schaefer, michael, associates}"
25,S1-404435514,Colline Molitor Sea LLC,"5620 Trinity Road, Unit 103, Raleigh, NC",US,"{molitor, colline, sea}"
26,S1-252242682,"Aggie E. Nagle, L.C.S.W.","11900 54th Avenue, Plymouth, MN",US,"{aggie, nagle}"
27,S1-981440385,Angad Multitrade Ltd,"Building No. 37/4, First Floor Opp Bharata Mat...",India,"{multitrade, angad}"
35,S1-945437722,Joeann Hills Park Inc.,"10101 Threave Road, Unit 301, Raleigh, NC",US,"{joeann, hills, park}"


In [35]:
sample_missing = missing_df[
    missing_df["has_index_token"]
].head(20)

display(
    sample_missing[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country",
            "name_tokens"
        ]
    ]
)

,entity_id,business_name,business_address,country,name_tokens
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,"{barbershop, orelee}"
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,"{prabhav, business}"
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India,"{consulting, private, nyasa, nursing}"
18,S1-221443075,Iris Advisors,House No-777 Raghubir Bhawanmadan Pur Khadar S...,India,"{iris, advisors}"
23,S1-716574008,Rajani Enterprises of Kannur,"46/1106, Dhanalakshmi Road, Near Amani Auditor...",India,"{enterprises, rajani, kannur}"
24,S1-985038901,"Schaefer, Michael and Silver Associates","32876 Circle Drive, Millsboro, DE",US,"{silver, schaefer, michael, associates}"
25,S1-404435514,Colline Molitor Sea LLC,"5620 Trinity Road, Unit 103, Raleigh, NC",US,"{molitor, colline, sea}"
26,S1-252242682,"Aggie E. Nagle, L.C.S.W.","11900 54th Avenue, Plymouth, MN",US,"{aggie, nagle}"
27,S1-981440385,Angad Multitrade Ltd,"Building No. 37/4, First Floor Opp Bharata Mat...",India,"{multitrade, angad}"
35,S1-945437722,Joeann Hills Park Inc.,"10101 Threave Road, Unit 301, Raleigh, NC",US,"{joeann, hills, park}"


In [34]:
import sqlite3

db_path = PROCESSED_DIR / "blocking_index.db"

conn_check = sqlite3.connect(db_path)

# Load all country + token combinations that survived blocking
token_rows = conn_check.execute(
    "SELECT DISTINCT country, token FROM token_index"
).fetchall()

conn_check.close()

token_index_set = set(token_rows)

print("Country-token combinations in blocking index:",
      len(token_index_set))

# Check the missed S1 entities
missing_df = s1[s1["entity_id"].isin(missing_true_s1)].copy()

def has_index_token(row):
    country = normalize_text(row["country"])
    tokens = row["name_tokens"]

    return any(
        (country, token) in token_index_set
        for token in tokens
    )

missing_df["has_index_token"] = missing_df.apply(
    has_index_token,
    axis=1
)

print()
print("Missed TRUE-match S1s:", len(missing_df))
print(
    "Missed S1s with at least one usable token in index:",
    missing_df["has_index_token"].sum()
)
print(
    "Missed S1s with NO token in index:",
    (~missing_df["has_index_token"]).sum()
)

Country-token combinations in blocking index: 1494137

Missed TRUE-match S1s: 978214
Missed S1s with at least one usable token in index: 443809
Missed S1s with NO token in index: 534405


In [33]:
missing_true_s1 = true_match_s1 - candidate_s1_new

print("TRUE-match S1s missed by current blocker:", len(missing_true_s1))
print("Percentage missed:", f"{1 - coverage_new:.2%}")

TRUE-match S1s missed by current blocker: 978214
Percentage missed: 46.95%


In [32]:
candidate_path = PROCESSED_DIR / "candidate_pairs.tsv"

candidate_s1_new = set()
candidate_rows_new = 0

for chunk in pd.read_csv(
    candidate_path,
    sep="\t",
    header=None,
    names=[
        "source1_entity_id",
        "candidate_entity_id",
        "candidate_source"
    ],
    dtype=str,
    chunksize=500_000
):
    candidate_rows_new += len(chunk)
    candidate_s1_new.update(chunk["source1_entity_id"].unique())

covered_new = true_match_s1 & candidate_s1_new
coverage_new = len(covered_new) / len(true_match_s1)

print("NEW CANDIDATE RESULTS")
print("=" * 40)
print("Candidate rows:", candidate_rows_new)
print("Unique S1 with candidates:", len(candidate_s1_new))
print("S1 entities with TRUE matches:", len(true_match_s1))
print("TRUE-match S1s with candidates:", len(covered_new))
print("S1 coverage:", f"{coverage_new:.2%}")
print(
    "Average candidates per S1:",
    candidate_rows_new / len(candidate_s1_new)
)

NEW CANDIDATE RESULTS
Candidate rows: 186873496
Unique S1 with candidates: 1161324
S1 entities with TRUE matches: 2083574
TRUE-match S1s with candidates: 1105360
S1 coverage: 53.05%
Average candidates per S1: 160.91417726663704


In [31]:
candidate_path = PROCESSED_DIR / "candidate_pairs.tsv"

candidate_s1_new = set()
candidate_rows_new = 0

for chunk in pd.read_csv(
    candidate_path,
    sep="\t",
    header=None,
    names=[
        "source1_entity_id",
        "candidate_entity_id",
        "candidate_source"
    ],
    dtype=str,
    chunksize=500_000
):
    candidate_rows_new += len(chunk)
    candidate_s1_new.update(chunk["source1_entity_id"].unique())

covered_new = true_match_s1 & candidate_s1_new

coverage_new = len(covered_new) / len(true_match_s1)

print("NEW CANDIDATE RESULTS")
print("=" * 40)
print("Candidate rows:", candidate_rows_new)
print("Unique S1 with candidates:", len(candidate_s1_new))
print("S1 entities with TRUE matches:", len(true_match_s1))
print("TRUE-match S1s with candidates:", len(covered_new))
print("S1 coverage:", f"{coverage_new:.2%}")
print(
    "Average candidates per S1:",
    candidate_rows_new / len(candidate_s1_new)
)

NEW CANDIDATE RESULTS
Candidate rows: 186873496
Unique S1 with candidates: 1161324
S1 entities with TRUE matches: 2083574
TRUE-match S1s with candidates: 1105360
S1 coverage: 53.05%
Average candidates per S1: 160.91417726663704


In [30]:
import pandas as pd

candidate_path = PROCESSED_DIR / "candidate_pairs.tsv"

header_check = pd.read_csv(
    candidate_path,
    sep="\t",
    nrows=5,
    dtype=str
)

print("Actual columns:")
print(header_check.columns.tolist())

print("\nFirst 5 rows:")
display(header_check)

Actual columns:
['S1-564644740', 'S2-620718070', 'source2']

First 5 rows:


,S1-564644740,S2-620718070,source2
0,S1-564644740,S3-401545450,source3
1,S1-564644740,S2-395669312,source2
2,S1-564644740,S2-730055352,source2
3,S1-564644740,S3-247121174,source3
4,S1-564644740,S3-679981840,source3


In [29]:
from pathlib import Path
import pandas as pd

candidate_path = PROCESSED_DIR / "candidate_pairs.tsv"

# Scan the NEW candidate file without loading all 74M+ rows into memory
candidate_s1_new = set()
candidate_rows_new = 0

for chunk in pd.read_csv(
    candidate_path,
    sep="\t",
    dtype={"source1_entity_id": str},
    usecols=["source1_entity_id"],
    chunksize=500_000
):
    candidate_rows_new += len(chunk)
    candidate_s1_new.update(chunk["source1_entity_id"].unique())

covered_new = true_match_s1 & candidate_s1_new

coverage_new = len(covered_new) / len(true_match_s1)

print("NEW CANDIDATE RESULTS")
print("=" * 40)
print("Candidate rows:", candidate_rows_new)
print("Unique S1 with candidates:", len(candidate_s1_new))
print("S1 entities with TRUE matches:", len(true_match_s1))
print("TRUE-match S1s with candidates:", len(covered_new))
print("S1 coverage:", f"{coverage_new:.2%}")
print(
    "Average candidates per S1:",
    candidate_rows_new / len(candidate_s1_new)
)

ValueError: Usecols do not match columns, columns expected but not found: ['source1_entity_id']

In [27]:
conn_test = sqlite3.connect(db_path)

distinct_tokens = conn_test.execute(
    "SELECT COUNT(DISTINCT token) FROM token_index"
).fetchone()[0]

print("Distinct tokens remaining:", distinct_tokens)

conn_test.close()

Distinct tokens remaining: 1432542


In [ ]:
import sqlite3

db_path = PROCESSED_DIR / "blocking_index.db"
conn_test = sqlite3.connect(db_path)

token_count = conn_test.execute(
    "SELECT COUNT(*) FROM token_index"
).fetchone()[0]

print("Remaining token-index rows:", token_count)

conn_test.close()


Remaining token-index rows: 8362018


In [25]:
print("S1 countries:")
print(s1["country"].value_counts().head(20))

print("\nS2 countries:")
s2_sample = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t",
    dtype=str,
    usecols=["country"],
    keep_default_na=False
)
print(s2_sample["country"].value_counts().head(20))

print("\nS3 countries:")
s3_sample = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t",
    dtype=str,
    usecols=["country"],
    keep_default_na=False
)
print(s3_sample["country"].value_counts().head(20))

S1 countries:
country
US       1323633
India     883188
Name: count, dtype: int64

S2 countries:
country
US       3016817
India    2017799
Name: count, dtype: int64

S3 countries:
country
US       3170056
India    2115547
Name: count, dtype: int64


In [24]:
print(
    no_candidate_s1["num_tokens"]
    .value_counts()
    .sort_index()
    .head(15)
)

num_tokens
0         40
1      69908
2     572559
3     871744
4     207974
5      19565
6       2527
7        415
8        117
9         47
10        16
11         8
12         3
13         1
14         1
Name: count, dtype: int64


In [23]:
s1 = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False
)

s1["norm_name"] = s1["business_name"].apply(normalize_text)
s1["name_tokens"] = s1["norm_name"].apply(get_tokens)

no_candidate_s1 = s1[
    ~s1["entity_id"].isin(candidate_s1)
].copy()

no_candidate_s1["num_tokens"] = no_candidate_s1["name_tokens"].apply(len)

print("S1 rows:", len(s1))
print("S1 with no candidates:", len(no_candidate_s1))

S1 rows: 2206821
S1 with no candidates: 1744926


In [22]:
print("TEST: notebook is running")
print("S1 rows:", len(s1))
print("S1 with no candidates:", len(no_candidate_s1))

TEST: notebook is running


NameError: name 's1' is not defined

In [21]:
import re
import pandas as pd

STOPWORDS = {
    "the", "and", "of", "for", "a", "an",
    "inc", "llc", "ltd", "limited",
    "corp", "corporation", "company", "co",
    "group", "center", "centre"
}

def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def get_tokens(text):
    return {
        token
        for token in text.split()
        if len(token) >= 2 and token not in STOPWORDS
    }

In [19]:
true_match_s1 = set(gt_pairs["source1_entity_id"])

candidate_s1 = set(candidates["source1_entity_id"])

covered_true_match_s1 = true_match_s1 & candidate_s1

print("S1 entities with TRUE matches:", len(true_match_s1))
print("S1 entities with candidates:", len(candidate_s1))
print(
    "S1 entities with TRUE matches that have candidates:",
    len(covered_true_match_s1)
)

coverage = len(covered_true_match_s1) / len(true_match_s1)

print(
    "S1 coverage among entities with true matches:",
    f"{coverage:.2%}"
)

S1 entities with TRUE matches: 2083574
S1 entities with candidates: 461895
S1 entities with TRUE matches that have candidates: 439451
S1 coverage among entities with true matches: 21.09%


In [18]:
# Create a clean ground-truth table containing one true pair per row

gt_pairs = ground_truth[
    ["source1_entity_id", "matched_entity_ids"]
].copy()

gt_pairs["matched_entity_ids"] = (
    gt_pairs["matched_entity_ids"]
    .fillna("")
    .str.strip()
)

# Remove singleton rows
gt_pairs = gt_pairs[gt_pairs["matched_entity_ids"] != ""]

# Convert comma-separated matches into individual rows
gt_pairs["matched_entity_ids"] = gt_pairs["matched_entity_ids"].str.split(",")

gt_pairs = gt_pairs.explode("matched_entity_ids")

gt_pairs["matched_entity_ids"] = (
    gt_pairs["matched_entity_ids"]
    .str.strip()
)

print("Total TRUE matching pairs:", len(gt_pairs))
print("S1 entities with true matches:", gt_pairs["source1_entity_id"].nunique())

display(gt_pairs.head(10))

Total TRUE matching pairs: 7638365
S1 entities with true matches: 2083574


,source1_entity_id,matched_entity_ids
0,S1-965667,S2-681193310
0,S1-965667,S2-743505751
0,S1-965667,S3-775321672
0,S1-965667,S3-11291185
0,S1-965667,S3-860443364
1,S1-55344266,S2-249013014
1,S1-55344266,S2-197070651
1,S1-55344266,S3-478195123
1,S1-55344266,S3-384364074
2,S1-343815751,S2-790675320


In [17]:
# Identify S1 entities that actually have at least one ground-truth match

ground_truth["has_match"] = (
    ground_truth["matched_entity_ids"]
    .fillna("")
    .str.strip()
    .ne("")
)

total_gt_s1 = len(ground_truth)
s1_with_true_matches = ground_truth["has_match"].sum()
s1_singletons = total_gt_s1 - s1_with_true_matches

print("Total ground-truth S1 entities:", total_gt_s1)
print("S1 entities with at least one TRUE match:", s1_with_true_matches)
print("S1 singleton entities:", s1_singletons)

print("\nPercentage with true matches:",
      round(100 * s1_with_true_matches / total_gt_s1, 2), "%")

print("Percentage singletons:",
      round(100 * s1_singletons / total_gt_s1, 2), "%")

Total ground-truth S1 entities: 2206821
S1 entities with at least one TRUE match: 2083574
S1 singleton entities: 123247

Percentage with true matches: 94.42 %
Percentage singletons: 5.58 %


In [16]:
train_s1_ids = set(source1["entity_id"])

candidate_s1_ids = set(candidates["source1_entity_id"])

print("Training S1 IDs:", len(train_s1_ids))
print("Candidate S1 IDs:", len(candidate_s1_ids))

print("\nCandidate IDs that are in training S1:",
      len(candidate_s1_ids & train_s1_ids))

print("Candidate IDs NOT in training S1:",
      len(candidate_s1_ids - train_s1_ids))

print("\nExample candidate S1 IDs:")
print(list(candidate_s1_ids)[:10])

Training S1 IDs: 2206821
Candidate S1 IDs: 461895

Candidate IDs that are in training S1: 461895
Candidate IDs NOT in training S1: 0

Example candidate S1 IDs:
['S1-907240357', 'S1-85233469', 'S1-376472263', 'S1-681864835', 'S1-402008915', 'S1-999633998', 'S1-622249976', 'S1-488406925', 'S1-111837302', 'S1-697024389']


In [15]:

source1_path = TRAIN_DIR / "train_source1.tsv"

source1 = pd.read_csv(
    source1_path,
    sep="\t",
    dtype=str,
    usecols=["entity_id"]
)

print("Total Source 1 entities:", len(source1))
print("S1 entities with candidates:", candidates["source1_entity_id"].nunique())
print(
    "S1 entities with NO candidates:",
    len(source1) - candidates["source1_entity_id"].nunique()
)

Total Source 1 entities: 2206821
S1 entities with candidates: 461895
S1 entities with NO candidates: 1744926


In [14]:
print("Number of candidate rows:", len(candidates))

print("Unique S1 entities:", candidates["source1_entity_id"].nunique())

print("Unique candidate entities:", candidates["candidate_entity_id"].nunique())

print(
    "Average candidates per S1:",
    len(candidates) / candidates["source1_entity_id"].nunique()
)

print("\nCandidates by source:")
print(candidates["candidate_source"].value_counts())

Number of candidate rows: 74147021
Unique S1 entities: 461895
Unique candidate entities: 5047202
Average candidates per S1: 160.52787105294493

Candidates by source:
candidate_source
source3    37946228
source2    36200793
Name: count, dtype: int64


In [13]:
candidates = pd.read_csv(
    candidate_path,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("Candidate shape:", candidates.shape)
print("Columns:", candidates.columns.tolist())

display(candidates.head(10))

Candidate shape: (74147021, 3)
Columns: ['source1_entity_id', 'candidate_entity_id', 'candidate_source']


,source1_entity_id,candidate_entity_id,candidate_source
0,S1-925783039,S2-252379152,source2
1,S1-925783039,S2-335180728,source2
2,S1-925783039,S3-534582038,source3
3,S1-925783039,S2-68778321,source2
4,S1-925783039,S2-141689655,source2
5,S1-925783039,S2-138574636,source2
6,S1-925783039,S2-597503035,source2
7,S1-925783039,S2-371520891,source2
8,S1-925783039,S3-794815197,source3
9,S1-925783039,S3-954751262,source3


In [12]:
ground_truth = pd.read_csv(
    ground_truth_path,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("Ground truth shape:", ground_truth.shape)
print("Columns:", ground_truth.columns.tolist())

display(ground_truth.head(10))

Ground truth shape: (2206821, 2)
Columns: ['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [11]:
TRAIN_DIR = PROJECT_ROOT / "data" / "raw" / "train"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

ground_truth_path = TRAIN_DIR / "train_ground_truth.tsv"
candidate_path = PROCESSED_DIR / "candidate_pairs.tsv"

print("Ground truth:", ground_truth_path)
print("Candidates:", candidate_path)

print("\nGround truth exists:", ground_truth_path.exists())
print("Candidates exist:", candidate_path.exists())

Ground truth: c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\raw\train\train_ground_truth.tsv
Candidates: c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\processed\candidate_pairs.tsv

Ground truth exists: True
Candidates exist: True


In [10]:
from pathlib import Path

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("RAW DATA:")
for file in RAW_DIR.rglob("*"):
    if file.is_file():
        print(" -", file)

print("\nPROCESSED DATA:")
for file in PROCESSED_DIR.rglob("*"):
    if file.is_file():
        print(" -", file)

RAW DATA:
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\raw\test\.gitkeep
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\raw\test\test_source1.tsv
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\raw\test\test_source2.tsv
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\raw\test\test_source3.tsv
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\raw\train\.gitkeep
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\raw\train\train_ground_truth.tsv
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\raw\train\train_source1.tsv
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\raw\train\train_source2.tsv
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\raw\train\train_source3.tsv

PROCESSED DATA:
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\processed\.gitkeep
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\d

In [9]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

print("Data directory:")
print(DATA_DIR)

print("\nContents of data:")
for item in DATA_DIR.iterdir():
    print(" -", item)

Data directory:
c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data

Contents of data:
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\processed
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data\raw


In [8]:
from pathlib import Path

current = Path.cwd()

print("Current notebook folder:")
print(current)

print("\nChecking parent folder:")
print(current.parent)

print("\nContents of parent folder:")
for item in current.parent.iterdir():
    print(" -", item)
    

Current notebook folder:
c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\notebooks

Checking parent folder:
c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26

Contents of parent folder:
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\.git
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\.gitattributes
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\.gitignore
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\models
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\notebooks
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\src


In [7]:
print("Training files:")

for file in TRAIN_DIR.iterdir():
    print(" -", file.name)

Training files:


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'c:\\Users\\chbha\\OneDrive\\Desktop\\Candidate gen\\AmazonML-26\\dataset\\train'

In [6]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
TRAIN_DIR = PROJECT_ROOT / "dataset" / "train"
OUTPUT_DIR = PROJECT_ROOT / "output"

print("Project root:", PROJECT_ROOT)
print("Training directory:", TRAIN_DIR)
print("Output directory:", OUTPUT_DIR)

Project root: c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26
Training directory: c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\dataset\train
Output directory: c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\output


In [5]:
from pathlib import Path

project_root = Path.cwd().parent

print("Project root:")
print(project_root)

print("\nContents of project root:")
for item in project_root.iterdir():
    print(" -", item)

Project root:
c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26

Contents of project root:
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\.git
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\.gitattributes
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\.gitignore
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\data
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\models
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\notebooks
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\src


In [4]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nFiles/folders here:")
for item in Path.cwd().iterdir():
    print(" -", item)

Current working directory:
c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\notebooks

Files/folders here:
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\notebooks\01_data_cleaning.ipynb
 - c:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\notebooks\candidate_evaluation.ipynb


In [2]:
from pathlib import Path

PROJECT_ROOT = Path("..")

TRAIN_DIR = PROJECT_ROOT / "dataset" / "train"
OUTPUT_DIR = PROJECT_ROOT / "output"

print("Training folder:", TRAIN_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())


Training folder: C:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\dataset\train
Output folder: C:\Users\chbha\OneDrive\Desktop\Candidate gen\AmazonML-26\output


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path